# District profile cleaning: Detect -> Judge -> Act
This notebook uses pandas and regex to inspect, clean and export only the district profile dataset. Run from the repository root. Input is preserved on first execution under `raw/before_cleaning/`; later runs replay that snapshot. No web requests or inferred demographic/contact values are introduced. The CDF datasets are not modified.


In [1]:
from pathlib import Path
import html
import re
import shutil
import pandas as pd

ROOT = Path.cwd()
filename = "db-unza26-csc4792-zimba_town_council_district_profile.csv"
target = ROOT / "data" / filename
snapshot = ROOT / "raw/before_cleaning" / filename
assert target.exists(), "Run from the repository root"
snapshot.parent.mkdir(parents=True, exist_ok=True)
if not snapshot.exists():
    shutil.copyfile(target, snapshot)
# Read as strings so phone prefixes and identifiers survive loading.
raw = pd.read_csv(snapshot, sep="|", dtype="string")
print("Input shape:", raw.shape)
print(raw.head().to_string(index=False))
raw.info()
print(raw.describe(include="all").to_string())
print("Missing values:\n", raw.isnull().sum())


Input shape: (4, 14)
   record_id        level      name population population_year population_male population_female households area_km2 province             neighboring_districts                                                                                                                                                                                                                 notes                                    source_url                         secondary_source_url
ZTC-DIST-001     district     Zimba      66725            2010           32186             34539      13284     5245 Southern Kalomo;Kazungula;Choma;Sinazongwe                                                                                                                   2010 Census by the Central Statistical Office (CSO); annual population growth rate reported as 2.9%  https://www.zimbacouncil.gov.zm/?page_id=759                                         <NA>
ZTC-DIST-002     district     Zimba      98533   

## Duplicates and text normalization: Detect
Inspect exact duplicates and repeated source URLs before making removal decisions. Decode HTML entities and collapse whitespace in ordinary text. Preserve URL punctuation, telephone prefixes and names; these are evidence fields, not a bag of words.


In [2]:
print("Exact duplicate rows:", raw.duplicated().sum())
print("Repeated source URLs:", raw.duplicated(subset=["source_url"]).sum())
clean = raw.copy()
def normalize(value):
    if pd.isna(value):
        return pd.NA
    return re.sub(r"\s+", " ", html.unescape(str(value))).strip() or pd.NA
for column in clean.columns:
    if column.endswith("url"):
        clean[column] = clean[column].str.strip().replace("", pd.NA)
    else:
        clean[column] = clean[column].map(normalize).astype("string")


Exact duplicate rows: 0
Repeated source URLs: 2


## Duplicates: Judge and Act
Three Zimba rows report 2010 census data, a 2018 projection and 2022 census data. Neither a shared URL nor name/level alone makes these duplicates. Remove only identical non-ID records and check name/level/year for conflicting observations. Preserve projection notes and report years so the observations are not misrepresented as equivalent census counts.


In [3]:
clean["level"] = clean["level"].str.lower().str.strip()
keys = [column for column in clean.columns if column != "record_id"]
before = len(clean)
clean = clean.drop_duplicates(subset=keys, keep="first").copy()
duplicates_removed = before - len(clean)
conflicts = clean[clean.duplicated(subset=["name", "level", "population_year"], keep=False)]
print("True duplicates removed:", duplicates_removed)
print("Conflicting name/level/year records:", len(conflicts))
assert conflicts.empty, "Review conflicting observations before export"


True duplicates removed: 0
Conflicting name/level/year records: 0


## Missing values and types: Detect -> Judge -> Act
Population and area are potential targets for demographic analyses; year is essential context for a population observation. Male/female counts and household counts are numeric supporting features or targets for other questions. Missing values are retained because the published constituency row is still useful as an administrative reference, and later-year district rows remain useful without sex/household breakdowns. Never copy a district total into a constituency row or carry 2010 counts forward to 2022. Province, neighbors, notes and secondary URLs are supporting features. A missing secondary URL means no secondary source is recorded, not a missing primary source.

Use `pd.to_numeric(errors='coerce')` and check whether conversion would discard a nonblank input. No outside-source supplementation is claimed. Missing demographic figures remain an explicitly documented limitation.


In [4]:
print(clean.isnull().sum())
numeric = ["population", "population_year", "population_male", "population_female", "households", "area_km2"]
for column in numeric:
    original = clean[column]
    values = pd.to_numeric(original, errors="coerce")
    assert not (original.notna() & values.isna()).any(), f"Review invalid {column}"
    assert (values.dropna() >= 0).all(), f"Review negative {column}"
    if column != "area_km2":
        assert (values.dropna() % 1 == 0).all(), f"Review fractional {column}"
        clean[column] = values.astype("Int64")
    else:
        clean[column] = values.astype("Float64")
assert clean.loc[clean["population"].notna(), "population_year"].notna().all()
assert clean["level"].isin(["district", "constituency", "ward"]).all()
both = clean[["population", "population_male", "population_female"]].notna().all(axis=1)
assert (clean.loc[both, "population_male"] + clean.loc[both, "population_female"] == clean.loc[both, "population"]).all()


record_id                0
level                    0
name                     0
population               1
population_year          1
population_male          3
population_female        3
households               3
area_km2                 1
province                 0
neighboring_districts    1
notes                    0
source_url               0
secondary_source_url     3
dtype: int64


## Outliers: Detect -> Judge -> Act
Compute 1.5-IQR fences separately by administrative level, excluding missing measurements and year identifiers. With only three district population observations, one sex/household observation and a repeated district area, these diagnostics are weak: no flags does not establish accuracy. Do not compare constituency and district populations as if they were the same measurement scale. A future flagged value requires checking its source; this notebook stops rather than removing or correcting it automatically.


In [5]:
iqr_rows = []
flagged_indices = set()
for level, group in clean.groupby("level"):
    for column in ["population", "population_male", "population_female", "households", "area_km2"]:
        values = group[column].dropna().astype(float)
        if values.empty:
            continue
        Q1, Q3 = values.quantile([0.25, 0.75])
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        flags = values[(values < lower) | (values > upper)]
        flagged_indices.update(flags.index)
        iqr_rows.append({"level": level, "column": column, "n": len(values), "lower": lower, "upper": upper, "flags": len(flags)})
print(pd.DataFrame(iqr_rows).to_string(index=False))
print("Flagged records:", len(flagged_indices))
assert not flagged_indices, "Review source evidence for flagged values before export"
outliers_removed = outliers_corrected = 0


   level            column  n    lower     upper  flags
district        population  3 51435.75 134617.75      0
district   population_male  1 32186.00  32186.00      0
district population_female  1 34539.00  34539.00      0
district        households  1 13284.00  13284.00      0
district          area_km2  3  5245.00   5245.00      0
Flagged records: 0


## Final validation and export
Preserve the original columns and source links. Optional punctuation removal/tokenization is omitted for this structured directory/profile: it would damage contacts and is unnecessary for the demographic measurements. Narrative notes remain intact. Export with `sep='|'` and `index=False`, retaining blanks, then reload as strings to check every exported field and schema. Existing identical files are not rewritten, allowing a file to remain open in a spreadsheet application.


In [6]:
assert clean["source_url"].notna().all(), "Review missing traceability URLs"
assert clean["source_url"].str.match(r"https?://", na=False).all()
assert clean["record_id"].notna().all() and clean["record_id"].is_unique
assert list(clean.columns) == list(raw.columns)
clean.info()
print(clean.describe(include="all").to_string())
exported = clean.to_csv(sep="|", index=False, na_rep="").encode("utf-8-sig")
if target.read_bytes() != exported:
    target.write_bytes(exported)
roundtrip = pd.read_csv(target, sep="|", dtype="string")
assert roundtrip.shape == clean.shape
pd.testing.assert_frame_equal(roundtrip.fillna(""), clean.astype("string").fillna(""), check_dtype=False)
print("Final rows:", len(clean), "columns:", len(clean.columns))
print("Duplicates removed:", duplicates_removed)
print("Final missing values:\n", clean.isnull().sum())
print("Saved:", filename)


<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   record_id              4 non-null      string 
 1   level                  4 non-null      string 
 2   name                   4 non-null      string 
 3   population             3 non-null      Int64  
 4   population_year        3 non-null      Int64  
 5   population_male        1 non-null      Int64  
 6   population_female      1 non-null      Int64  
 7   households             1 non-null      Int64  
 8   area_km2               3 non-null      Float64
 9   province               4 non-null      string 
 10  neighboring_districts  3 non-null      string 
 11  notes                  4 non-null      string 
 12  source_url             4 non-null      string 
 13  secondary_source_url   1 non-null      string 
dtypes: Float64(1), Int64(5), string(8)
memory usage: 604.0 bytes
           r

## Handoff summary
The district profile retains four rows and 14 columns, including the separate 2010 census, 2018 projection and 2022 census observations. No true duplicates are removed, and no IQR outliers are corrected or removed; the small sample limits outlier inference. Missing constituency measurements and later-year sex/household breakdowns remain blank rather than estimated. Notes and source URLs are preserved, and integer measurements use nullable integer types during processing.
